# 02 · Model training and comparison
Two tasks for the BharatAssist guard layer:
1. **Safety classification** – is a user prompt *safe* or *unsafe*? (binary)
2. **Language identification** – which of 12 languages is it in? (multi-class)

**Features:** character n-gram TF-IDF (2–4 characters). Character n-grams work across all 12
scripts without a separate tokenizer for each language.

**Models** (all subclasses of the abstract `BaseTextClassifier`): majority baseline, Multinomial
Naive Bayes, Logistic Regression, Linear SVM, Random Forest.

**Protocol:** train on *train*, choose models and settings on *valid*, report final numbers once on *test*.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

from bharatassist.config import SAMPLE_DIR, FIGURE_DIR, MODEL_DIR, REPORT_DIR

In [2]:
data = {s: pd.read_csv(SAMPLE_DIR.parent / "clean" / f"{s}_clean.csv.gz") for s in ("train", "valid", "test")}
tr, va, te = data["train"], data["valid"], data["test"]
Xtr, Xva, Xte = tr["clean_prompt"], va["clean_prompt"], te["clean_prompt"]
print({s: len(d) for s, d in data.items()})

{'train': 47992, 'valid': 11986, 'test': 17996}


## Task 1 · Safety classification – compare models on the validation set

In [3]:
from bharatassist.models import all_models
from bharatassist.evaluation import binary_metrics

models, rows = {}, []
for model in all_models():          # polymorphism: same calls for every model
    model.fit(Xtr, tr["label"])
    pred = model.predict(Xva)
    metrics = binary_metrics(va["label"], pred, model.decision_scores(Xva))
    rows.append({"Model": model.name, **metrics, "Train time (s)": model.train_seconds})
    models[model.name] = model
    print(f"{model.name:20s} macro-F1 = {metrics['Macro F1']:.3f}")

val_table = pd.DataFrame(rows).set_index("Model").round(3)
val_table

Majority baseline    macro-F1 = 0.375


Naive Bayes          macro-F1 = 0.678


Logistic Regression  macro-F1 = 0.738


Linear SVM           macro-F1 = 0.740


Random Forest        macro-F1 = 0.713


,Accuracy,Precision (unsafe),Recall (unsafe),F1 (unsafe),Macro F1,ROC-AUC,Train time (s)
Model,,,,,,,
Majority baseline,0.600,0.600,1.000,0.750,0.375,0.500,21.622
Naive Bayes,0.707,0.720,0.839,0.775,0.678,0.773,23.227
Logistic Regression,0.746,0.803,0.764,0.783,0.738,0.821,29.739
Linear SVM,0.748,0.802,0.769,0.785,0.740,0.821,25.555
Random Forest,0.731,0.756,0.814,0.784,0.713,0.793,97.159


In [4]:
from bharatassist.evaluation import grouped_metric_chart
grouped_metric_chart(val_table, ["Accuracy", "Macro F1", "Recall (unsafe)"],
                     "Safety models – validation set", path=FIGURE_DIR / "safety_model_comparison.png")
plt.show()

The **majority baseline** shows why accuracy misleads: about 60 % accuracy but macro-F1 of only
0.375, because it never predicts "safe".

### Tuning the regularisation strength *C* (validation set)

In [5]:
from bharatassist.models import LinearSVMModel, LogisticRegressionModel
tuning = []
for C in (0.1, 0.25, 0.5, 1.0):
    m = LinearSVMModel(C=C).fit(Xtr, tr["label"])
    tuning.append({"Model": "Linear SVM", "C": C, "Macro F1": binary_metrics(va["label"], m.predict(Xva))["Macro F1"]})
tuning = pd.DataFrame(tuning).round(4)
display(tuning)
best_C = float(tuning.loc[tuning["Macro F1"].idxmax(), "C"])
print("Best C for Linear SVM:", best_C)

,Model,C,Macro F1
0,Linear SVM,0.10,0.7329
1,Linear SVM,0.25,0.7396
2,Linear SVM,0.50,0.7395
3,Linear SVM,1.00,0.7355


Best C for Linear SVM: 0.25


### Over-fitting check – train vs validation

In [6]:
fit_rows = []
for name in ("Naive Bayes", "Logistic Regression", "Linear SVM", "Random Forest"):
    m = models[name]
    fit_rows.append({"Model": name,
                     "Train macro-F1": binary_metrics(tr["label"], m.predict(Xtr))["Macro F1"],
                     "Valid macro-F1": val_table.loc[name, "Macro F1"]})
fit_table = pd.DataFrame(fit_rows).set_index("Model")
fit_table["Gap"] = fit_table["Train macro-F1"] - fit_table["Valid macro-F1"]
fit_table.round(3)

,Train macro-F1,Valid macro-F1,Gap
Model,,,
Naive Bayes,0.722,0.678,0.044
Logistic Regression,0.844,0.738,0.106
Linear SVM,0.885,0.740,0.145
Random Forest,0.953,0.713,0.240


All models score higher on training data than on validation data. The Random Forest has the
largest gap (it memorises the training set). Linear models generalise best.

### Final safety model – evaluated once on the test set

In [7]:
best = LinearSVMModel(C=best_C).fit(Xtr, tr["label"])
test_pred = best.predict(Xte)
test_metrics = pd.Series(binary_metrics(te["label"], test_pred, best.decision_scores(Xte))).round(3)
test_metrics.to_frame("Linear SVM – test")

,Linear SVM – test
Accuracy,0.732
Precision (unsafe),0.783
Recall (unsafe),0.750
F1 (unsafe),0.766
Macro F1,0.727
ROC-AUC,0.803


In [8]:
from bharatassist.evaluation import confusion_chart
confusion_chart(te["label"].values, test_pred, ["safe", "unsafe"], "Safety – confusion matrix (test)",
                label_values=[0, 1], path=FIGURE_DIR / "safety_confusion.png")
plt.show()

In [9]:
from bharatassist.evaluation import per_group_f1, bar_h
lang_f1 = per_group_f1(te, test_pred, "language")
bar_h(lang_f1, "Safety model macro-F1 by language (test)", fmt="{:.2f}", xlim=(0, 1),
      highlight=lang_f1.idxmax(), path=FIGURE_DIR / "safety_f1_by_language.png")
plt.show()
tag_f1 = per_group_f1(te, test_pred, "tag")
tag_f1.round(3).to_frame("macro-F1 (test)")

,macro-F1 (test)
tag,
Jailbreak,0.687
Culture-adaptive,0.724
Generic,0.736


## Task 2 · Language identification (12 classes)

In [10]:
from bharatassist.models import NaiveBayesModel
from bharatassist.evaluation import multiclass_metrics
lang_rows, lang_models = [], {}
for model in (NaiveBayesModel(), LinearSVMModel(C=0.5, balanced=False)):
    model.fit(Xtr, tr["language"])
    pred = model.predict(Xva)
    lang_rows.append({"Model": model.name, **multiclass_metrics(va["language"], pred),
                      "Train time (s)": model.train_seconds})
    lang_models[model.name] = model
lang_table = pd.DataFrame(lang_rows).set_index("Model").round(4)
lang_table

,Accuracy,Macro precision,Macro recall,Macro F1,Train time (s)
Model,,,,,
Naive Bayes,0.9978,0.9979,0.9978,0.9978,23.0873
Linear SVM,0.9985,0.9985,0.9985,0.9985,31.7965


In [11]:
lang_best = lang_models["Linear SVM"]
lang_test_pred = lang_best.predict(Xte)
print("Test accuracy:", round(multiclass_metrics(te["language"], lang_test_pred)["Accuracy"], 4))
names = sorted(te["language"].unique())
confusion_chart(te["language"].values, lang_test_pred, names, "Language ID – confusion matrix (test)",
                normalize=True, figsize=(6.5, 5.8), path=FIGURE_DIR / "language_confusion.png")
plt.show()
errors = te.assign(pred=lang_test_pred).query("language != pred")
print("Most common confusions:")
errors.groupby(["language", "pred"]).size().sort_values(ascending=False).head(5).to_frame("count")

Test accuracy: 0.9986
Most common confusions:


,,count
language,pred,
Marathi,English,6
Hindi,Marathi,4
Marathi,Hindi,2
Punjabi,English,2
Telugu,English,2


## Save models and export the Excel report

In [12]:
from bharatassist.evaluation import export_excel
p1 = best.save(MODEL_DIR / "safety_model.pkl")
p2 = lang_best.save(MODEL_DIR / "language_model.pkl")
for p in (p1, p2):
    print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")

xlsx = export_excel({
    "Safety - validation": val_table,
    "Safety - C tuning": tuning.set_index("Model"),
    "Safety - overfitting": fit_table.round(3),
    "Safety - test": test_metrics.to_frame("Linear SVM"),
    "Safety - by language": lang_f1.round(3).to_frame("macro F1"),
    "Language ID": lang_table,
}, REPORT_DIR / "model_comparison.xlsx")
print("Excel report:", xlsx.name)

safety_model.pkl: 4.7 MB
language_model.pkl: 13.5 MB
Excel report: model_comparison.xlsx


### Threshold analysis
The SVM flags a prompt as unsafe when its score is above 0. Raising the threshold blocks fewer
safe prompts (higher precision) but lets more unsafe ones through (lower recall).

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score
scores = best.decision_scores(Xte)
thr_rows = []
for t in (-0.5, -0.25, 0.0, 0.25, 0.5, 0.75):
    p = (scores > t).astype(int)
    thr_rows.append({"Threshold": t, "Precision (unsafe)": precision_score(te["label"], p),
                     "Recall (unsafe)": recall_score(te["label"], p),
                     "Macro F1": f1_score(te["label"], p, average="macro"),
                     "% prompts blocked": p.mean() * 100})
thr_table = pd.DataFrame(thr_rows).set_index("Threshold").round(3)
thr_table

,Precision (unsafe),Recall (unsafe),Macro F1,% prompts blocked
Threshold,,,,
-0.50,0.650,0.958,0.592,86.241
-0.25,0.708,0.890,0.689,73.466
0.00,0.783,0.750,0.727,56.001
0.25,0.850,0.547,0.678,37.614
0.50,0.908,0.331,0.571,21.310
0.75,0.951,0.154,0.444,9.491


### Error analysis – domain-shift test on BharatAssist-style questions
The real app receives questions about government schemes. Let us check how the model scores
harmless in-domain questions.

In [14]:
from bharatassist.models import BaseTextClassifier
from bharatassist.preprocessing import TextCleaner
safety = BaseTextClassifier.load(MODEL_DIR / "safety_model.pkl")
language = BaseTextClassifier.load(MODEL_DIR / "language_model.pkl")
cleaner = TextCleaner()

in_domain = [
    "How do I apply for the PM-KISAN scheme?",
    "What documents do I need for Ayushman Bharat?",
    "Can you help me fill my pension form?",
    "पीएम किसान योजना के लिए आवेदन कैसे करें?",
    "ஆயுஷ்மான் பாரத் அட்டை எப்படி பெறுவது?",
    "Hello",
    "How can I hack into my neighbour's WiFi?",
]
cleaned = [cleaner.clean_strict(t) for t in in_domain]
scores_d = safety.decision_scores(cleaned)
pd.DataFrame({"prompt": in_domain, "language": language.predict(cleaned),
              "score": scores_d.round(2), "verdict": np.where(scores_d > 0, "unsafe", "safe")})

,prompt,language,score,verdict
0,How do I apply for the PM-KISAN scheme?,English,0.48,unsafe
1,What documents do I need for Ayushman Bharat?,English,-0.09,safe
2,Can you help me fill my pension form?,English,-0.13,safe
3,पीएम किसान योजना के लिए आवेदन कैसे करें?,Hindi,0.82,unsafe
4,ஆயுஷ்மான் பாரத் அட்டை எப்படி பெறுவது?,Tamil,0.14,unsafe
5,Hello,English,-1.03,safe
6,How can I hack into my neighbour's WiFi?,English,1.12,unsafe


In [15]:
eng = tr[tr["language"] == "English"]
gov = eng[eng["clean_prompt"].str.contains("scheme|yojana|government", regex=True)]
print(f"English training prompts mentioning scheme/yojana/government: {len(gov)}")
print(f"Share labelled unsafe: {gov['label'].mean():.0%}   (all English prompts: {eng['label'].mean():.0%})")

English training prompts mentioning scheme/yojana/government: 65
Share labelled unsafe: 80%   (all English prompts: 57%)


**Finding 6 – domain shift.** In IndicGuard, prompts that mention schemes or the government are
mostly unsafe (hate, misinformation and manipulation prompts *about* government). The model has
therefore learnt that scheme vocabulary is risky, and it **wrongly blocks some harmless scheme
questions** – exactly the questions BharatAssist users ask. The last row (a real unsafe request)
is correctly blocked.

**Planned fix (Review-2):**
1. Collect real, safe in-domain questions – scrape scheme FAQs from official portals with
   `requests` + `BeautifulSoup` (Unit 3) – and add them to training.
2. Build a small in-domain validation set and **calibrate the threshold** on it.
3. Compare against a Keras neural network (Unit 4).

## Summary
| Task | Best model | Test result |
|---|---|---|
| Safety (safe vs unsafe) | Linear SVM, char n-gram TF-IDF | see test table above |
| Language ID (12 classes) | Linear SVM | ~99.9 % accuracy |

* Linear models beat Naive Bayes and Random Forest; Random Forest over-fits the most.
* Jailbreak prompts are the hardest prompt type for the safety model.
* Out-of-domain false positives on scheme questions must be fixed before deployment.